<center> <h1 style="background-color:orange; color:white"><br>Exploratory Data Analysis<br></h1></center>

# `Problem Statement:`
We have used Cars dataset from kaggle  with features including make, model, year, engine, and other properties of the car used to predict its price.

## `Importing the necessary libraries`





In [ ]:
# Import the libraries used in this analysis
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## `Load the dataset into dataframe`

In [ ]:
# Load the original public Kaggle Cars dataset.
# Use a local/Colab upload when supplied, otherwise download the public dataset.
import os
from pathlib import Path

dataset_path = os.getenv("CARS_DATASET_PATH")
if dataset_path:
    csv_path = Path(dataset_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"CARS_DATASET_PATH does not exist: {csv_path}")
else:
    csv_path = Path("Cars_data.csv")

if csv_path.is_file():
    df = pd.read_csv(csv_path)
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
        if csv_name is not None:
            csv_path = Path(csv_name)
            df = pd.read_csv(csv_path)
        else:
            raise FileNotFoundError("No CSV file was uploaded.")
    except ImportError:
        import kagglehub

        dataset_dir = Path(kagglehub.dataset_download("CooperUnion/cardataset"))
        candidates = list(dataset_dir.rglob("*.csv"))
        required = {"Make", "Model", "Year", "Engine HP", "MSRP"}
        selected = None
        for candidate in candidates:
            try:
                columns = set(pd.read_csv(candidate, nrows=0).columns)
            except Exception:
                continue
            if required.issubset(columns):
                selected = candidate
                break
        if selected is None:
            raise FileNotFoundError(
                "Could not locate the expected Cars dataset CSV. "
                "Set CARS_DATASET_PATH to the original CSV."
            )
        csv_path = selected
        df = pd.read_csv(csv_path)

if "df" not in globals():
    df = pd.read_csv(csv_path)

required_columns = {"Make", "Model", "Year", "Engine HP", "MSRP"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Cars dataset is missing required columns: {sorted(missing_columns)}")

print(f"Loaded dataset: {csv_path}")
print("Dataset shape:", df.shape)
df.head()


In [ ]:
# Get the datatypes of each column
df.dtypes


Now we observe the each features present in the dataset.<br>

 `Make:` The Make feature is the company name of the Car.<br>
`Model:` The Model feature is the model or different version of Car models.<br>
`Year:`  The year describes the model has been launched.<br>
`Engine Fuel Type:` It defines the Fuel type of the car model.<br>
`Engine HP:` It's say the Horsepower that refers to the power an engine produces.<br>
`Engine Cylinders:` It define the nos of cylinders in present in the engine.<br>
`Transmission Type:` It is the type of feature that describe about the car transmission type i.e Mannual or automatic.<br>
`Driven_Wheels:` The type of wheel drive.<br>
`No of doors:` It defined nos of doors present in the car.<br>
`Market Category:` This features tells about the type of car or which category the car belongs. <br>
`Vehicle Size:` It's say about the about car size.<br>
`Vehicle Style:` The feature is all about the style that belongs to car.<br>
`highway MPG:` The average a car will get while driving on an open stretch of road without stopping or starting, typically at a higher speed.<br>
`city mpg:` City MPG refers to driving with occasional stopping and braking.<br>
`Popularity:` It can refered to rating of that car or popularity of car.<br>
`MSRP:` The price of that car.







## `Check the datatypes`

In [ ]:
# Preserve the full source dataset before restricting the charting scope.
from src.car_eda import data_quality_flags, validate_cars
validate_cars(df)
df_original = df.copy(deep=True)
print("Original dataset shape:", df_original.shape)
print("Duplicate rows in original data:", int(df_original.duplicated().sum()))
print("Missing values by column:")
display(df_original.isna().sum().to_frame("missing_count"))
quality_flags = data_quality_flags(df_original)
print("Data-quality flags (review before exclusion):")
display(quality_flags.to_frame())
# Preserve source data; the analysis dataframe is created from a copy below.

## `Dropping irrevalent columns`

The notebook keeps a focused set of fields for the visual analysis below. The original dataset also contains fuel type, market category, vehicle style, popularity, number of doors, and vehicle size. These fields are excluded from the current chart sequence for scope and readability, not because they are inherently irrelevant or harmful to model accuracy. This notebook performs exploratory analysis only. It does not train or evaluate a prediction model.

In [ ]:
# Initialise cols_to_drop
cols_to_drop = ["Engine Fuel Type", "Market Category", "Vehicle Style",
                "Popularity", "Number of Doors", "Vehicle Size"]


In [ ]:
# Keep the original dataset untouched and create a focused analysis copy.
df = df_original.drop(columns=cols_to_drop, errors="raise").copy()
print("Original columns retained in df_original:", len(df_original.columns))
print("Columns in focused EDA dataframe:", len(df.columns))
df.head()

## `Renaming the columns`

Now, Its time for renaming the feature to useful feature name. It will help to use them in model training purpose.<br>

We have already dropped the unneccesary columns, and now we are left with useful columns. One extra thing that we would do is to rename the columns such that the name clearly represents the essence of the column.

The given dict represents (in key value pair) the previous name, and the new name for the dataframe columns

In [ ]:
# Rename columns using a clear and useful naming convention
rename_cols = {
    "Engine HP": "HP",
    "Engine Cylinders": "Cylinders",
    "Driven_Wheels": "Drive Mode",
    "MSRP": "Price"
}


In [ ]:
# Use pandas rename function
df = df.rename(columns=rename_cols)
df.head()


In [ ]:
# Print the head of the dataframe
df.head()


## `Dropping the duplicate rows`

There are many rows in the dataframe which are duplicate, and hence they are just repeating the information. Its better if we remove these rows as they don't add any value to the dataframe. 

For given data, we would like to see how many rows were duplicates. For this, we will count the number of rows, remove the dublicated rows, and again count the number of rows.

In [ ]:
# Number of rows before removing duplicated rows
print("Number of rows before removing duplicates:", len(df))


In [ ]:
# Drop exact duplicated rows using a reusable, tested helper.
from src.car_eda import remove_exact_duplicates
rows_before_deduplication = len(df)
df = remove_exact_duplicates(df)
print("Rows before duplicate removal:", rows_before_deduplication)
print("Duplicate rows removed:", rows_before_deduplication - len(df))
print("Rows after duplicate removal:", len(df))
df.head()

In [ ]:
# Count number of rows after deleting duplicated rows
print("Number of rows after removing duplicates:", len(df))


## `Dropping the null or missing values`

Missing values are usually represented in the form of Nan or null or None in the dataset.

Finding whether we have null values in the data is by using the isnull() function.

There are many values which are missing, in pandas dataframe these values are reffered to as np.nan. We want to deal with these values beause we can't use nan values to train models. Either we can remove them to apply some strategy to replace them with other values.

To keep things simple we will be dropping nan values

In [ ]:
# Check for NaN values in each column
df.isnull().sum()


The saved dataset output shows missing values in Engine HP and Engine Cylinders. We remove rows with missing values for this exploratory exercise. In a predictive modelling workflow, compare this choice with train-only imputation and assess whether dropping rows introduces bias.

In [ ]:
# Complete-case analysis for this EDA. The original dataset remains in df_original.
rows_before_missing = len(df)
missing_by_row = df.isna().any(axis=1)
df = df.loc[~missing_by_row].copy()
rows_removed_missing = rows_before_missing - len(df)

print("Rows before missing-value removal:", rows_before_missing)
print("Rows removed:", rows_removed_missing)
print("Rows remaining:", len(df))
print("Missing values remaining:", int(df.isna().sum().sum()))


In [ ]:
# Make sure that missing values are removed
df.isnull().sum()


In [ ]:
# Describe statistics of df
df.describe()


## `Removing outliers`

Sometimes a dataset can contain extreme values that are outside the range of what is expected and unlike the other data. These are called outliers and often machine learning modeling and model skill in general can be improved by understanding and even removing these outlier values.

In [ ]:
# Plot a boxplot for 'Price' column
plt.figure(figsize=(8, 5))
sns.boxplot(x=df["Price"], color="steelblue")
plt.title("Boxplot of Car Price")
plt.xlabel("Price")
plt.show()


### Observation

The box plot highlights the skewed distribution of car prices and potential high-price outliers. A box-plot outlier is a statistical flag, not proof of an incorrect record. Luxury and high-performance vehicles can have valid high prices.

In [ ]:
# Plot a boxplot for 'HP' column
plt.figure(figsize=(8, 5))
sns.boxplot(x=df["HP"], color="steelblue")
plt.title("Boxplot of Horsepower (HP)")
plt.xlabel("HP")
plt.show()


### Observation

The horsepower distribution contains values beyond the central range. Review these observations against vehicle context before deciding whether to exclude them.

print all the columns which are of int or float datatype in df. 

Hint: Use loc with condition

In [ ]:
# Correct numeric-column selection. Use NumPy's numeric dtype selector.
numeric_df = df.select_dtypes(include=[np.number])
print("Numeric columns:", numeric_df.columns.tolist())
print("Numeric dataframe shape:", numeric_df.shape)
display(numeric_df.head())

### `Save the column names of the above output in variable list named 'l'`


In [ ]:
# Save numeric column names in variable list named 'l'.
l = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", l)


## **`Outliers removal techniques - IQR Method`**
 

**Here comes cool Fact for you!**

IQR is the first quartile subtracted from the third quartile; these quartiles can be clearly seen on a box plot on the data.

- Calculate IQR  and give a suitable threshold to remove the outliers and save this new dataframe into df2.

Let us help you to decide threshold: Outliers in this case are defined as the observations that are below (Q1 − 1.5x IQR) or above (Q3 + 1.5x IQR)

In [ ]:
# Use IQR as a sensitivity analysis on HP and Price only.
# Keep the primary cleaned dataframe unchanged.
from src.car_eda import iqr_sensitivity_filter
outlier_features = ("HP", "Price")
df2, outlier_mask = iqr_sensitivity_filter(df, columns=outlier_features)
Q1 = df[list(outlier_features)].quantile(0.25); Q3 = df[list(outlier_features)].quantile(0.75)
IQR = Q3 - Q1; lower_bound = Q1 - 1.5 * IQR; upper_bound = Q3 + 1.5 * IQR
print("Features checked:", list(outlier_features))
print("Q1:"); display(Q1)
print("Q3:"); display(Q3)
print("IQR:"); display(IQR)
print("Lower bounds:"); display(lower_bound)
print("Upper bounds:"); display(upper_bound)
print("Rows flagged:", int(outlier_mask.sum()))
print("Rows retained in df2:", len(df2))
print("Share flagged (%):", round(outlier_mask.mean() * 100, 2))

In [ ]:
# Compare the original cleaned dataframe with the sensitivity-analysis copy
print("Cleaned dataframe shape:", df.shape)
print("HP/Price IQR-filtered dataframe shape:", df2.shape)
print("Rows excluded from df2:", len(df) - len(df2))


In [ ]:
# Find unique values and their counts in each column in df
for i in df.columns:
    print("--------------- %s ----------------" % i)
    print(df[i].value_counts(dropna=False))


## `Visualising Univariate Distributions`

We will use seaborn library to visualize eye catchy univariate plots. 

Do you know? you have just now already explored one univariate plot. guess which one? Yeah its box plot.


### `Histogram & Density Plots`

Histograms and density plots show the frequency of a numeric variable along the y-axis, and the value along the x-axis. The ```sns.distplot()``` function plots a density curve. Notice that this is aesthetically better than vanilla ```matplotlib```.

In [ ]:
# Plotting distribution for variable HP
plt.figure(figsize=(8, 5))
sns.histplot(df["HP"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of HP")
plt.xlabel("HP")
plt.ylabel("Frequency")
plt.show()


### Observation

The histogram and density curve show the distribution of horsepower. The distribution is right-skewed, so the mean can be influenced by high-horsepower vehicles. Use the median and quantiles alongside the mean when describing a typical car.

Since seaborn uses matplotlib behind the scenes, the usual matplotlib functions work well with seaborn. For example, you can use subplots to plot multiple univariate distributions.
- Hint: use matplotlib subplot function

In [ ]:
# Plot all columns present in list l using subplot of dimension (2,3)
c = 0
plt.figure(figsize=(15, 10))

for i in l:
    c += 1
    plt.subplot(2, 3, c)
    sns.histplot(df[i], kde=True, bins=30, color="steelblue")
    plt.title(i)
    plt.xlabel(i)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()


## `Bar Chart Plots`


Plot a histogram depicting the make in X axis and number of cars in y axis. <br>

In [ ]:
# Plot Make on X axis and number of cars on Y axis
plt.figure(figsize=(12, 8))

make_counts = df["Make"].value_counts().nlargest(10)
make_counts.plot(kind="bar")

plt.title("Top 10 Car Makes by Number of Cars")
plt.xlabel("Make")
plt.ylabel("Number of Cars")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Observation

This chart ranks the ten most frequent car manufacturers (Make) in the dataset. It shows the composition of the dataset, not the market share of all cars sold.

### `Count Plot`
A count plot can be thought of as a histogram across a categorical, instead of quantitative, variable.


 Plot a countplot for a variable Transmission vertically with hue as Drive mode

In [ ]:
# Plot countplot for Transmission and Drive Mode
plt.figure(figsize=(15, 5))
sns.countplot(data=df, x="Transmission Type", hue="Drive Mode", palette="deep")
plt.title("Transmission Type by Drive Mode")
plt.xlabel("Transmission Type")
plt.ylabel("Count")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### Observation

The count plot compares transmission categories and splits each category by drive mode. Counts describe the records in this dataset and should not be interpreted as population-wide sales figures.

# `Visualising Bivariate Distributions`


Bivariate distributions are simply two univariate distributions plotted on x and y axes respectively. They help you observe the relationship between the two variables.




## `Scatter Plots`
Scatterplots are used to find the correlation between two continuos variables.

Using scatterplot find the correlation between 'HP' and 'Price' column of the data. 



In [ ]:
# Scatterplot between HP and Price
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(data=df, x="HP", y="Price", ax=ax, color="steelblue")
ax.set_title("HP vs Price")
ax.set_xlabel("HP")
ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

print("Correlation between HP and Price:", df["HP"].corr(df["Price"]))


### Observation

The scatter plot compares horsepower with MSRP. The saved Pearson correlation is approximately 0.660, indicating a positive linear association in this dataset. Correlation does not establish that horsepower causes a higher price, and the relationship can be influenced by make, model, year, and vehicle segment.

## `Plotting Aggregated Values across Categories`


### `Bar Plots - Mean, Median and Count Plots`



Bar plots are used to **display aggregated values** of a variable, rather than entire distributions. This is especially useful when you have a lot of data which is difficult to visualise in a single figure. 

For example, say you want to visualise and *compare the Price across Cylinders*. The ```sns.barplot()``` function can be used to do that.


In [ ]:
# Bar plot with default statistic = mean between Cylinders and Price
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x="Cylinders", y="Price", color="steelblue")
plt.title("Average Price by Number of Cylinders")
plt.xlabel("Cylinders")
plt.ylabel("Average Price")
plt.tight_layout()
plt.show()


### **`Observation:`**<br>
By default, seaborn plots the mean value across categories, though you can plot the count, median, sum etc.<br>
Also, barplot computes and shows the confidence interval of the mean as well.



## `When you want to visualise having a large number of categories, it is helpful to plot the categories across the y-axis.`

### `Let's now drill down into Transmission sub categories.`

In [ ]:
# Plot categorical variable Transmission across the y-axis
plt.figure(figsize=(10, 6))
sns.countplot(data=df, y="Transmission Type", color="steelblue")
plt.title("Number of Cars by Transmission Type")
plt.xlabel("Number of Cars")
plt.ylabel("Transmission Type")
plt.tight_layout()
plt.show()


These plots looks beutiful isn't it? In Data Analyst life such charts are there unavoidable friend.:)

# `Multivariate Plots`



## `Heatmaps`


A heat map is a two-dimensional representation of information with the help of colors. Heat maps can help the user visualize simple or complex information

Using heatmaps plot the correlation between the features present in the dataset.

In [ ]:
# Find the correlation of numeric features
corr = df.select_dtypes(include=["number"]).corr()
print(corr)


In [ ]:
# Plot the heatmap using the correlated dataframe
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="BrBG", annot=True, fmt=".2f")
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()


### Interpretation

The heatmap summarizes pairwise Pearson correlations among numeric columns. Values range from -1 to +1. Strong correlations describe linear association, not causation. Correlation can also be affected by outliers, mixed vehicle segments, and confounding variables. Review scatter plots and group-level patterns before drawing business conclusions.

## Final Note

This notebook performs exploratory data analysis on the Kaggle Cars dataset. It covers data inspection, duplicate and missing-value handling, univariate and bivariate analysis, category summaries, and numeric correlations.

The IQR-filtered dataframe is a separate sensitivity-analysis copy based on horsepower and MSRP. The original cleaned dataframe remains unchanged for the main EDA. The notebook does not build a machine-learning model and does not establish causal relationships.